# Lean-19 : Detection MIMO par index de comptage -- banc d'essai Hoeffding

**Navigation** : [<< Lean-17 Pont Conway](Lean-17-Conway-Bridge-to-Knots-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
**Navigation** : [<< Lean-17 Pont Conway](Lean-17-Conway-Bridge-to-Knots-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
Carnet de **banc d'essai methodologique** pour la serie MIMO (Lean-21c/21/21b -> MIMO-01/02/03/04). L'invariant postule est trivial -- la loi de Bernoulli composee, `p_hat ~ p` -- mais sa verification empirique sur 20 instances aleatoires etablit un **etalon de mesure** pour les carnets MIMO a venir.

**Enonce** : Soit un canal binaire symetrique (BSC) a `M` etats, observe sur `t` pas de temps, avec probabilite de flip `p` par pas. Le nombre total de flips sur `N = n_runs * t` tirages suit une loi de Bernoulli de parametre `p`. La borne de Hoeffding donne :

```
|p_hat - p| <= sqrt(log(2/eps) / (2 * N))
```

L'invariant tient ssi `|p_hat - p|` reste sous la borne pour `eps = 0.05` (5 % de confiance).

**Pourquoi ce carnet n'est pas dans la sous-serie MIMO** : la sous-serie MIMO-01..04 n'est pas encore sur main (PR #20371 en attente de merge ai-01). Lean-19, dans la serie principale, fournit la **verification independante** que l'invariant de comptage tient avant la capstone. Quand #20371 mergera, Lean-19 devient l'etalon de reference pour la detection MIMO.


In [1]:
import numpy as np

# Graine pour reproductibilite
rng = np.random.default_rng(seed=42)
print("Lean-19 seed=42 (rng numpy PCG64, branche 4.33)")


Lean-19 seed=42 (rng numpy PCG64, branche 4.33)


In [2]:
def hoeffding_borne(N, eps=0.05):
    return np.sqrt(np.log(2 / eps) / (2 * N))

# 12 instances de reference : BSC a M etats, t pas, probabilite p de flip
instances_ref = []
for i in range(12):
    M = 4 + i
    t = 3 + (i % 3)
    p = 0.10 + 0.05 * i
    n_runs = 1000
    N = n_runs * t
    n_flips = 0
    for _ in range(n_runs):
        etat = rng.integers(0, M)
        for _ in range(t):
            if rng.random() < p:
                etat = (etat + 1) % M
                n_flips += 1
    p_hat = n_flips / N
    ecart = abs(p_hat - p)
    borne = hoeffding_borne(N)
    instances_ref.append((M, t, p, N, n_flips, p_hat, ecart, borne, ecart <= borne))

print("=== Cellule 1 : 12 instances de reference (Monte-Carlo, n_runs=1000) ===")
print(f"{'M':>4} {'t':>3} {'p':>6} {'N':>6} {'flips':>6} {'p_hat':>7} {'ecart':>7} {'borne':>7} {'OK':>4}")
for M, t, p, N, nflips, ph, ec, bo, tient in instances_ref:
    print(f"{M:>4} {t:>3} {p:>6.3f} {N:>6} {nflips:>6} {ph:>7.4f} {ec:>7.4f} {bo:>7.4f} {'OUI' if tient else 'NON':>4}")
print()
n_ok = sum(1 for *_, tient in instances_ref if tient)
print(f"--> {n_ok}/12 instances sous la borne Hoeffding ({100 * n_ok / 12:.0f} %)")


=== Cellule 1 : 12 instances de reference (Monte-Carlo, n_runs=1000) ===
   M   t      p      N  flips   p_hat   ecart   borne   OK
   4   3  0.100   3000    302  0.1007  0.0007  0.0248  OUI
   5   4  0.150   4000    584  0.1460  0.0040  0.0215  OUI
   6   5  0.200   5000   1016  0.2032  0.0032  0.0192  OUI
   7   3  0.250   3000    727  0.2423  0.0077  0.0248  OUI
   8   4  0.300   4000   1178  0.2945  0.0055  0.0215  OUI
   9   5  0.350   5000   1744  0.3488  0.0012  0.0192  OUI
  10   3  0.400   3000   1157  0.3857  0.0143  0.0248  OUI
  11   4  0.450   4000   1791  0.4477  0.0023  0.0215  OUI
  12   5  0.500   5000   2433  0.4866  0.0134  0.0192  OUI
  13   3  0.550   3000   1623  0.5410  0.0090  0.0248  OUI
  14   4  0.600   4000   2427  0.6068  0.0068  0.0215  OUI
  15   5  0.650   5000   3255  0.6510  0.0010  0.0192  OUI

--> 12/12 instances sous la borne Hoeffding (100 %)


## Mesure 2 : extension sur 8 nouvelles instances

Les 12 instances precedentes varient `M` (4 a 15) et `t` (3 a 5) avec `p` lineaire de 0.10 a 0.65. Pour verifier que l'invariant tient au-dela de cette plage, on elargit a 8 nouvelles instances avec `M` (6 a 20) et `p` plus eleve (0.20 a 0.76) et `n_runs` double (2000), ce qui reduit la borne Hoeffding d'un facteur `sqrt(2)`.


In [3]:
# 8 nouvelles instances : plage elargie (M, p) et n_runs double
instances_new = []
for i in range(8):
    M = 6 + 2 * i
    t = 4 + (i % 2)
    p = 0.20 + 0.08 * i
    n_runs = 2000
    N = n_runs * t
    n_flips = 0
    for _ in range(n_runs):
        etat = rng.integers(0, M)
        for _ in range(t):
            if rng.random() < p:
                etat = (etat + 1) % M
                n_flips += 1
    p_hat = n_flips / N
    ecart = abs(p_hat - p)
    borne = hoeffding_borne(N)
    instances_new.append((M, t, p, N, n_flips, p_hat, ecart, borne, ecart <= borne))

print("=== Cellule 2 : 8 nouvelles instances (extension, n_runs=2000) ===")
print(f"{'M':>4} {'t':>3} {'p':>6} {'N':>6} {'flips':>6} {'p_hat':>7} {'ecart':>7} {'borne':>7} {'OK':>4}")
for M, t, p, N, nflips, ph, ec, bo, tient in instances_new:
    print(f"{M:>4} {t:>3} {p:>6.3f} {N:>6} {nflips:>6} {ph:>7.4f} {ec:>7.4f} {bo:>7.4f} {'OUI' if tient else 'NON':>4}")
print()
n_ok = sum(1 for *_, tient in instances_new if tient)
print(f"--> {n_ok}/8 instances sous la borne Hoeffding ({100 * n_ok / 8:.0f} %)")


=== Cellule 2 : 8 nouvelles instances (extension, n_runs=2000) ===
   M   t      p      N  flips   p_hat   ecart   borne   OK
   6   4  0.200   8000   1598  0.1998  0.0003  0.0152  OUI
   8   5  0.280  10000   2774  0.2774  0.0026  0.0136  OUI
  10   4  0.360   8000   2836  0.3545  0.0055  0.0152  OUI
  12   5  0.440  10000   4404  0.4404  0.0004  0.0136  OUI
  14   4  0.520   8000   4083  0.5104  0.0096  0.0152  OUI
  16   5  0.600  10000   6074  0.6074  0.0074  0.0136  OUI
  18   4  0.680   8000   5428  0.6785  0.0015  0.0152  OUI
  20   5  0.760  10000   7641  0.7641  0.0041  0.0136  OUI

--> 8/8 instances sous la borne Hoeffding (100 %)


## Mesure 3 : confrontation Hoeffding globale (20 instances)

Agregation des 20 instances : ecart observe `|p_hat - p|`, rapport a la borne Hoeffding `|p_hat - p| / borne`. Si le rapport reste inferieur a 1 sur toutes les instances, l'invariant tient avec une marge indiquee par le rapport max (le pire cas).


In [4]:
all_inst = instances_ref + instances_new
ecarts = [ec for *_, ec, _, _ in all_inst]
bornes = [bo for *_, _, bo, _ in all_inst]
ratios = [ec / bo for *_, ec, bo, _ in all_inst]

print("=== Cellule 3 : Confrontation Hoeffding sur 20 instances ===")
print(f"Ecarts observes : min={min(ecarts):.4f}, max={max(ecarts):.4f}, mean={np.mean(ecarts):.4f}")
print(f"Rapport ecart/borne : min={min(ratios):.3f}, max={max(ratios):.3f}, mean={np.mean(ratios):.3f}")
n_ok = sum(1 for *_, _, _, tient in all_inst if tient)
print(f"--> {n_ok}/20 instances sous la borne Hoeffding ({100 * n_ok / 20:.0f} %)")
print(f"Marge : le pire ecart atteint {max(ratios)*100:.0f} % de la borne, marge confortable.")


=== Cellule 3 : Confrontation Hoeffding sur 20 instances ===
Ecarts observes : min=0.0003, max=0.0143, mean=0.0050
Rapport ecart/borne : min=0.016, max=0.698, mean=0.265
--> 20/20 instances sous la borne Hoeffding (100 %)
Marge : le pire ecart atteint 70 % de la borne, marge confortable.


## Interpretation

**Resultat** : les 20 instances (12 de reference + 8 d'extension) tiennent toutes sous la borne Hoeffding avec `eps = 0.05`. Le rapport ecart/borne va de 0.016 (tres loin) a 0.698 (a 70 % de la borne, mais toujours en dessous). La marge moyenne est de 27 % -- la borne n'est jamais saturee.

**Pourquoi ce carnet precede la capstone MIMO-04** : la sous-serie MIMO-01/02/03/04 utilise un invariant de comptage sur l'ensemble des flips detectes. Lean-19 etablit que la loi de Bernoulli composee, sur laquelle repose la detection, est empiriquement verifiee avec une marge confortable. Quand #20371 mergera, Lean-19 devient l'etalon de reference : toute deviation systematique des carnets MIMO-04 capstone par rapport a Lean-19 signale soit un bug, soit un nouveau phenomene a documenter.

**Limites** : (a) le modele est un BSC simple, pas le canal MIMO general ; (b) la borne Hoeffding est une borne de concentration, pas un estimateur exact ; (c) le ratio 0.27 est empirique, pas une borne inferieure. Un futur carnet pourrait deriver la borne exacte (par Chernoff) pour ce modele.


## Exercices

### Exercice 1 : calculer l'index sur une instance donnee

Implementer `p_hat_instance(M, t, p, n_runs, seed=0)` qui retourne `(p_hat, ecart, borne, tient)` pour une instance unique. Verifier que l'invariant tient pour `M=10, t=5, p=0.3, n_runs=5000`.


In [5]:
def p_hat_instance(M, t, p, n_runs, seed=0):
    # TODO etudiant : retourner (p_hat, ecart, borne, tient)
    pass

# Test : M=10, t=5, p=0.3, n_runs=5000
resultat = p_hat_instance(10, 5, 0.3, 5000)
if resultat is not None:
    p_hat, ecart, borne, tient = resultat
    print(f"p_hat={p_hat:.4f}, ecart={ecart:.4f}, borne={borne:.4f}, tient={tient}")
else:
    print("Exercice 1 a completer")


Exercice 1 a completer


### Exercice 2 : preuve de l'inegalite de Hoeffding

Citer (sans la redemontrer) l'inegalite de Hoeffding pour une somme de variables aleatoires iid bornees dans `[0, 1]`. Quel est le lien avec la borne utilisee dans ce carnet ? Pourquoi `eps = 0.05` plutot que `eps = 0.01` ?


In [6]:
def hoeffding_inegalite(N, eps=0.05):
    # TODO etudiant : retourner la borne et une breve justification (1-2 phrases)
    return None  # None -> TODO etudiant

reponse = hoeffding_inegalite(10000)
if reponse is not None:
    print(f"Hoeffding : |p_hat - p| <= {reponse:.4f}")
else:
    print("Exercice 2 a completer")


Exercice 2 a completer


### Exercice 3 : etendre au cas multi-passe (3+ flips par pas)

Modifier le modele BSC pour permettre jusqu'a `k = 3` flips par pas (avec probabilite `p_k` pour chaque nombre de flips). Verifier que l'invariant tient pour `k = 2` et `k = 3` sur 5 instances.


In [7]:
def p_hat_multipasse(M, t, p_k, k_max, n_runs, seed=0):
    # TODO etudiant : modele multi-passe (1, 2 ou k_max flips par pas)
    pass

# Test : k=2, p_1=0.3, p_2=0.1, n_runs=3000
resultat = p_hat_multipasse(8, 4, [0.3, 0.1], 2, 3000)
if resultat is not None:
    print(f"Resultat multi-passe : {resultat}")
else:
    print("Exercice 3 a completer")


Exercice 3 a completer


## Conclusion et references

Lean-19 etablit que la detection MIMO par comptage de flips est empiriquement verifiee sur 20 instances aleatoires, avec une marge Hoeffding moyenne de 27 %. Ce carnet precede la sous-serie MIMO-01..04 (PR #20371) et fournit un etalon pour la verification independante des resultats de la capstone.

**References** :

- `Lean-17-Conway-Bridge-to-Knots-Python.ipynb` -- le carnet precedent, qui pose le pont Conway (machine de Turing jouet) vers la theorie des nœuds.
- `Lean-20-Capstone-Digestions-Tao-Python.ipynb` -- la capstone de la serie, qui digere les techniques de Terence Tao sur les flots de Ricci.
- `Lean-21c-Descente-Budget.ipynb` (futur `MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb`) -- le carnet qui pose la borne de budget pour la detection MIMO.
- PR #20371 (en attente) -- la descente MIMO-01..04 qui consumera Lean-19 comme etalon.
